### Hospital Bed Occupancy and Resource Utilization

In [1]:
import pandas as pd
import numpy as np

##### Datasets

In [2]:
patients = pd.read_csv("datasets_raw/patients.csv")
services = pd.read_csv("datasets_raw/services_weekly.csv")
staff = pd.read_csv("datasets_raw/staff.csv")
staff_schedule = pd.read_csv("datasets_raw/staff_schedule.csv")

In [3]:
patients.head()

,patient_id,name,age,arrival_date,departure_date,service,satisfaction
0,PAT-09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61
1,PAT-f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83
2,PAT-ac6162e4,Julia Torres,24,2025-06-29,2025-07-05,general_medicine,83
3,PAT-3dda2bb5,Crystal Johnson,32,2025-10-12,2025-10-23,emergency,81
4,PAT-08591375,Garrett Lin,25,2025-02-18,2025-02-25,ICU,76


In [4]:
patients.describe()

,age,satisfaction
count,1000.000000,1000.000000
mean,45.337000,79.597000
std,25.999912,11.550325
min,0.000000,60.000000
25%,23.000000,70.000000
50%,46.000000,80.000000
75%,68.000000,89.250000
max,89.000000,99.000000


In [5]:
patients.shape

(1000, 7)

In [6]:
patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   patient_id      1000 non-null   object
 1   name            1000 non-null   object
 2   age             1000 non-null   int64 
 3   arrival_date    1000 non-null   object
 4   departure_date  1000 non-null   object
 5   service         1000 non-null   object
 6   satisfaction    1000 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 54.8+ KB


In [7]:
services.head()

,week,month,service,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,event
0,1,1,emergency,32,76,32,44,67,70,none
1,1,1,surgery,45,130,45,85,83,78,flu
2,1,1,general_medicine,37,201,37,164,97,43,flu
3,1,1,ICU,22,31,22,9,84,91,flu
4,2,1,emergency,28,169,28,141,75,64,none


In [8]:
services.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   week                  208 non-null    int64 
 1   month                 208 non-null    int64 
 2   service               208 non-null    object
 3   available_beds        208 non-null    int64 
 4   patients_request      208 non-null    int64 
 5   patients_admitted     208 non-null    int64 
 6   patients_refused      208 non-null    int64 
 7   patient_satisfaction  208 non-null    int64 
 8   staff_morale          208 non-null    int64 
 9   event                 208 non-null    object
dtypes: int64(8), object(2)
memory usage: 16.4+ KB


In [9]:
services.describe()

,week,month,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale
count,208.000000,208.000000,208.000000,208.000000,208.000000,208.000000,208.000000,208.000000
mean,26.500000,6.923077,30.346154,64.870192,28.129808,36.740385,80.000000,72.567308
std,15.044539,3.634755,15.172929,58.738572,14.676791,55.015763,11.125546,15.457759
min,1.000000,1.000000,8.000000,5.000000,5.000000,0.000000,60.000000,31.000000
25%,13.750000,4.000000,18.000000,23.750000,16.000000,0.000000,70.000000,60.000000
50%,26.500000,7.000000,27.500000,49.000000,26.000000,13.500000,81.000000,73.000000
75%,39.250000,10.000000,40.000000,86.000000,37.000000,52.500000,89.000000,86.000000
max,52.000000,12.000000,74.000000,388.000000,74.000000,363.000000,99.000000,99.000000


In [10]:
services.shape

(208, 10)

In [11]:
staff.head()

,staff_id,staff_name,role,service
0,STF-5ca26577,Allison Hill,doctor,emergency
1,STF-02ae59ca,Noah Rhodes,doctor,emergency
2,STF-d8006e7c,Angie Henderson,doctor,emergency
3,STF-212d8b31,Daniel Wagner,doctor,emergency
4,STF-107a58e4,Cristian Santos,doctor,emergency


In [12]:
staff.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   staff_id    110 non-null    object
 1   staff_name  110 non-null    object
 2   role        110 non-null    object
 3   service     110 non-null    object
dtypes: object(4)
memory usage: 3.6+ KB


In [13]:
staff.describe(include='all')

,staff_id,staff_name,role,service
count,110,110,110,110
unique,110,110,3,4
top,STF-5ca26577,Allison Hill,nurse,ICU
freq,1,1,69,32


In [14]:
staff.shape

(110, 4)

In [15]:
staff_schedule.head()

,week,staff_id,staff_name,role,service,present
0,1,STF-b77cdc60,Allison Hill,doctor,emergency,1
1,2,STF-b77cdc60,Allison Hill,doctor,emergency,1
2,3,STF-b77cdc60,Allison Hill,doctor,emergency,0
3,4,STF-b77cdc60,Allison Hill,doctor,emergency,1
4,5,STF-b77cdc60,Allison Hill,doctor,emergency,1


In [16]:
staff_schedule.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6552 entries, 0 to 6551
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   week        6552 non-null   int64 
 1   staff_id    6552 non-null   object
 2   staff_name  6552 non-null   object
 3   role        6552 non-null   object
 4   service     6552 non-null   object
 5   present     6552 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 307.3+ KB


In [17]:
staff_schedule.describe()

,week,present
count,6552.000000,6552.000000
mean,26.500000,0.599817
std,15.009476,0.489973
min,1.000000,0.000000
25%,13.750000,0.000000
50%,26.500000,1.000000
75%,39.250000,1.000000
max,52.000000,1.000000


In [18]:
staff_schedule.shape

(6552, 6)

##### Checking missing values

In [19]:
patients.isnull().sum()

patient_id        0
name              0
age               0
arrival_date      0
departure_date    0
service           0
satisfaction      0
dtype: int64

In [20]:
services.isnull().sum()

week                    0
month                   0
service                 0
available_beds          0
patients_request        0
patients_admitted       0
patients_refused        0
patient_satisfaction    0
staff_morale            0
event                   0
dtype: int64

In [21]:
staff.isnull().sum()

staff_id      0
staff_name    0
role          0
service       0
dtype: int64

In [22]:
staff_schedule.isnull().sum()

week          0
staff_id      0
staff_name    0
role          0
service       0
present       0
dtype: int64

##### Check Duplicate Records

In [23]:
patients.duplicated().sum()

np.int64(0)

In [24]:
services.duplicated().sum()

np.int64(0)

In [25]:
staff.duplicated().sum()

np.int64(0)

In [26]:
staff_schedule.duplicated().sum()

np.int64(0)

##### Check unique values

In [27]:
patients['service'].unique()

array(['surgery', 'general_medicine', 'emergency', 'ICU'], dtype=object)

In [28]:
staff['service'].unique()

array(['emergency', 'surgery', 'general_medicine', 'ICU'], dtype=object)

In [29]:
staff['role'].unique()

array(['doctor', 'nurse', 'nursing_assistant'], dtype=object)

In [30]:
services['service'].unique()

array(['emergency', 'surgery', 'general_medicine', 'ICU'], dtype=object)

In [31]:
services['event'].unique()

array(['none', 'flu', 'donation', 'strike'], dtype=object)

In [32]:
staff_schedule['present'].unique()

array([1, 0])

##### Converting to date

In [34]:
patients['arrival_date'] = pd.to_datetime(patients['arrival_date'], dayfirst=True)

patients['departure_date'] = pd.to_datetime(patients['departure_date'], dayfirst=True)

##### Checking if it has invalid dates

In [35]:
patients[patients['departure_date'] < patients['arrival_date']]

,patient_id,name,age,arrival_date,departure_date,service,satisfaction


##### Checking Numerical Columns

In [36]:
services[services['patients_refused'] > services['patients_request']]

,week,month,service,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,event


In [37]:
services[services['patients_refused'] > services['patients_request']]

,week,month,service,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,event


#### Feature Engineering

##### Patients

###### LOS

In [38]:
patients['length_of_stay'] = (patients['departure_date'] - patients['arrival_date']).dt.days

###### Admission Month

In [40]:
patients['admission_month'] = (patients['arrival_date'].dt.month_name())


###### Admission Quarter

In [41]:
patients['admission_quarter'] = (patients['arrival_date'].dt.quarter)

###### Day of Week

In [42]:
patients['admission_day'] = (patients['arrival_date'].dt.day_name())

###### Age Groups

In [43]:
patients['age_group'] = pd.cut(patients['age'],bins=[0,18,35,50,65,120], labels=['0-18','19-35','36-50','51-65','65+'])

##### Services Weekly

###### Bed Utilization %

In [44]:
services['bed_utilization_pct'] = (
    services['patients_admitted'] /
    services['available_beds']
) * 100

###### Admission Rate %

In [45]:
services['admission_rate_pct'] = (
    services['patients_admitted'] /
    services['patients_request']
) * 100

###### Refusal Rate %

In [46]:
services['refusal_rate_pct'] = (
    services['patients_refused'] /
    services['patients_request']
) * 100

###### Demand Pressure

In [47]:
services['demand_pressure'] = (
    services['patients_request'] /
    services['available_beds']
)

##### Staff

In [48]:
staff['role'] = staff['role'].str.title()

staff['service'] = staff['service'].str.title()

##### Staff Schedule

In [49]:
staff_schedule['attendance_status'] = (
    staff_schedule['present']
    .map({
        1:'Present',
        0:'Absent'
    })
)

#### Saving cleaned data

In [50]:
patients.to_csv(
    "patients_clean.csv",
    index=False
)

services.to_csv(
    "services_weekly_clean.csv",
    index=False
)

staff.to_csv(
    "staff_clean.csv",
    index=False
)

staff_schedule.to_csv(
    "staff_schedule_clean.csv",
    index=False
)